# Introduction to qim3d
Introduction to **qim3d**,  a Python library for **Quantitative Imaging in 3D**.

In this notebook we will:

- Generate a **single synthetic volume** (an organic blob) with `qim3d.generate.volume`.
- Generate a small **synthetic dataset** of multiple objects with `qim3d.generate.volume_collection`.
- Explore the data with qim3d's visualization tools: slice grids, interactive slicers, histograms, and 3D ray-cast rendering with `qim3d.viz.volumetric`.

In [ ]:
import qim3d

# Random seed for reproducibility
SEED = 42

## 1. Generating a single synthetic volume

`qim3d.generate.volume` creates a synthetic 3D object based on structured Perlin noise, so the result looks like an organic structure (a cell, a tissue blob, a pore).

Key parameters:

- `base_shape` / `final_shape`: output resolution.
- `noise_scale`: texture roughness — lower values are smoother, higher values are more detailed.
- `threshold`: cut-off defining the object surface — lower values give larger objects.
- `shape`: `None` (blob), `'cylinder'`, or `'tube'`.

In [ ]:
# Generate a single organic blob
blob = qim3d.generate.volume(
    base_shape=(128, 128, 128),
    noise_scale=0.03,
    seed=SEED,
)

print(f"Shape: {blob.shape}")
print(f"Dtype: {blob.dtype}")
print(f"Intensity range: {blob.min()} .. {blob.max()}")

In [ ]:
qim3d.viz.slices_grid(blob, n_slices=15)

In [ ]:
qim3d.viz.volumetric(blob)

In [ ]:
qim3d.viz.export_rotation('blob.gif', blob, show=True)

In [ ]:
qim3d.viz.line_profile(blob)

## 2. Generating a synthetic dataset

`qim3d.generate.volume_collection` places multiple non-overlapping objects inside a larger container volume, and returns a **label mask** where each object has a unique integer ID. This is ideal for creating training data for instance segmentation or object detection.


In [ ]:
volume_collection, labels = qim3d.generate.volume_collection(
    n_volumes=32,
    collection_shape=(256, 256, 256),
    shape_range=((32, 32, 32), (64, 64, 64)),
    noise_range=(0.02, 0.04),
    seed=SEED,
)

print(f"Collection shape: {volume_collection.shape}")

In [ ]:
qim3d.viz.slices_grid(volume_collection, n_slices=15)

In [ ]:
qim3d.viz.volumetric(volume_collection)

## 3. Checking the labels

In this case, labels are directly created by qim3d, but they could be the result of a segmentation task. Here, we can easily explore and compare labels with the original volume.

In [ ]:
cmap = qim3d.viz.colormaps.segmentation(n_labels=labels.max())
qim3d.viz.slices_grid(labels, colormap=cmap, max_value=labels.max(), n_slices=15)

In [ ]:
qim3d.viz.overlay(volume_collection, labels, colormaps=('viridis', cmap), 
                  volume2_values=(0, labels.max()))